# `docx_renderer` — render & inspect

Renders a `.docx` to page PNGs with the offline engine and shows the pages inline, plus a few
checks that read the engine's own layout model rather than the pixels.

**To run:** open this notebook in VS Code, click **Select Kernel** in the top right, and pick the
project's `.venv`. If VS Code offers to install `ipykernel`, accept it — that is the only extra
package the notebook needs; everything else is already in the venv.

## 1. Configure

In [ ]:
from pathlib import Path
import sys, time

# The notebook lives at the repo root, next to the `docx_renderer` package. The package is
# imported from the source tree, so the repo root has to be on sys.path (`pip install -e .`
# would make this unnecessary).
PROJECT = Path.cwd()
if not (PROJECT / "docx_renderer").is_dir():
    PROJECT = next(p for p in PROJECT.parents if (p / "docx_renderer").is_dir())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

DOCX = PROJECT / "NDA BLA XXXXXX - Safety Tables and Figures - 2 Columns RD.docx"
OUT = PROJECT / "inspect_out"          # git-ignored
DPI = 110                              # 110 reads well inline; bump to 150+ to pixel-peep

# Which pages to rasterize: "3", "1-4", "1,3,5-7", "8-", or None for the whole document.
# Layout always runs over the whole document, so this only trades render time for coverage
# (~4s for four pages of the 88-page sample, ~19s for all of it).
PAGES = "1-4"

# LLM visual-grounding overlays (see README) — all off for a clean render.
SHOW_RULER = False    # 0.5" tab ruler across the top + vertical tab grid lines
SHOW_BBOXES = False   # color-coded boxes around paragraphs, tables, cells, list markers
EXPORT_JSON = False   # companion JSON mapping every XML XPath -> page bbox

assert DOCX.exists(), f"not found: {DOCX}"
print("python :", sys.executable)
print("docx   :", DOCX.name)

## 2. Render to PNGs

In [ ]:
from docx_renderer import render_docx_to_png

t0 = time.time()
pages = render_docx_to_png(
    docx_path=DOCX,
    output_dir=OUT,
    prefix="page",
    dpi=DPI,
    show_ruler=SHOW_RULER,
    show_bboxes=SHOW_BBOXES,
    export_layout_json=EXPORT_JSON,
    pages=PAGES,
)
print(f"{len(pages)} pages in {time.time() - t0:.1f}s -> {OUT}")
for p in pages:
    print("  ", p.name)

## 3. View pages inline

`show(1, 2, 3)` displays those page numbers; `show_range(1, 4)` is the same for a span. PNGs are
named after the page they actually are, so `page_9.png` is page 9 whether or not pages 1-8 were
rendered — asking for a page you skipped in `PAGES` just says so.

In [ ]:
from IPython.display import display, HTML, Image as IPyImage

def page_png(n: int) -> Path:
    return OUT / f"page_{n}.png"

def show(*nums: int, width: int = 760) -> None:
    for n in nums:
        p = page_png(n)
        if not p.exists():
            print(f"page {n}: no such file ({p.name})")
            continue
        display(HTML(f"<div style='margin-top:14px;font:600 13px system-ui'>page {n} &mdash; {p.name}</div>"))
        display(IPyImage(filename=str(p), width=width))

def show_range(first: int, last: int, width: int = 760) -> None:
    show(*range(first, last + 1), width=width)

In [ ]:
# The front matter: title page + Table of Contents, then the Table of Tables / Table of Figures.
show_range(1, 4)

## 4. Checks on the two TOC bugs

Both were fixed on 2026-10-26. These cells assert against the engine's resolved model, so they
stay meaningful regardless of DPI or anti-aliasing.

| | bug | fix |
|---|---|---|
| 1 | "Table of Tables" / "Table of Figures" entries rendered blue + underlined | `parser.py` — `DocxParser.is_toc_entry_style()` |
| 2 | TOC page numbers wrapped to their own line, dot leaders dropped | `layout.py` — tab stops outside the right indent are honored |
| 3 | every table rendered as a solid black block | `parser.py` — `w:fill="auto"` means *no* fill |
| 4 | a blank page 11, with the real page 11 pushed to 12 | `layout.py` — no hardcoded minimum table row height |

Run the pipeline by hand so the intermediate objects are available (~2s — the slow part above is
rasterizing to PNG, not parsing or laying out).

In [ ]:
from docx_renderer import DocxParser, LayoutEngine, FontManager

fm = FontManager()
parser = DocxParser(DOCX)
doc = parser.parse()
layout_pages = LayoutEngine(doc, font_manager=fm).layout_document()
print(f"{len(doc.blocks)} top-level blocks -> {len(layout_pages)} laid-out pages")

### Check 1 — TOC entries must not be styled as hyperlinks

Word writes `<w:rStyle w:val="Hyperlink"/>` on every entry of a `TOC \h` field but renders those
entries in the entry style's own color. The engine suppresses the blue + underline for any
paragraph style in the TOC family — `toc 1`..`toc 9`, `table of figures`, `table of authorities`,
and anything based on them.

Expect every TOC entry run to come out **black with no underline**.

In [ ]:
from docx_renderer.model import ParagraphBlock, TableBlock

# Yield every ParagraphBlock, descending into table rows and cells.
def iter_paragraphs(blocks):
    for b in blocks:
        if isinstance(b, ParagraphBlock):
            yield b
        elif isinstance(b, TableBlock):
            for row in b.rows:
                for cell in row.cells:
                    yield from iter_paragraphs(cell.blocks)

all_paras = list(iter_paragraphs(doc.blocks))
print(f"{len(all_paras)} paragraphs (including inside tables)")

In [ ]:
toc_paras = [p for p in all_paras if parser.is_toc_entry_style(p.style_id)]
styles_seen = sorted({p.style_id for p in toc_paras})
looks = {(r.style.color, r.style.underline) for p in toc_paras for r in p.runs if r.kind == "text"}

print(f"TOC entry paragraphs : {len(toc_paras)}  ({', '.join(styles_seen)})")
print(f"(color, underline)   : {looks}")

blue = {lk for lk in looks if lk != ("000000", None)}
print("\nPASS - no TOC entry is styled as a hyperlink" if not blue else f"\nFAIL - {blue}")

In [ ]:
# The contrast: genuine hyperlinks elsewhere in the document are still blue + underlined.
def link_runs(p):
    return [r for r in p.runs
            if r.kind == "text" and r.style.underline and r.style.color not in (None, "000000")]

real = [(p, link_runs(p)) for p in all_paras if not parser.is_toc_entry_style(p.style_id)]
real = [(p, rs) for p, rs in real if rs]

print(f"{len(real)} paragraphs with a real hyperlink still rendering blue + underlined:\n")
for p, rs in real[:8]:
    txt = "".join(r.text for r in p.runs if r.kind == "text")[:52]
    print(f"  {str(p.style_id):<16} #{rs[0].style.color} u={rs[0].style.underline:<8} {txt!r}")

### Check 2 — dot leaders and right-aligned page numbers

`TOC1` declares a right tab with `leader="dot"` at 8640 twips (432 pt, i.e. 6.0" from the left
margin), and the TOC styles also set a right indent — which puts that tab stop *outside* the text
boundary. Word measures tab stops from the left margin and honors them anyway; the engine now does
too, instead of wrapping the page number onto its own line and dropping the leader with it.

Expect one dot leader per TOC entry, each ending just short of 72 pt (margin) + 432 pt = **504 pt**,
the small gap being the width of the page number that follows.

In [ ]:
from collections import Counter

leaders = [
    (pg.page_number, bb.style_id, bb.summary, bb.x0, bb.x1)
    for pg in layout_pages for bb in pg.bboxes if bb.kind == "tab_leader"
]
print(f"dot leaders drawn : {len(leaders)}  (TOC entry paragraphs: {len(toc_paras)})")
print("by style          :", dict(Counter(s for _, s, _, _, _ in leaders)))
print("leader ends at x  :", dict(Counter(round(x1) for *_, x1 in leaders)), "pt")

ok = len(leaders) == len(toc_paras) and all(485 <= x1 <= 504 for *_, x1 in leaders)
print("\nPASS - every entry has a leader running to the 6.0 inch tab stop" if ok else "\nFAIL")

### Check 3 — table cells must not be filled black

`<w:shd w:val="clear" w:color="auto" w:fill="auto"/>` is what Word writes for an unshaded cell.
`auto` defers the choice to the renderer, and the two attributes resolve differently: automatic
*text* (`w:color`) is black, but an automatic *fill* (`w:fill`) is the absence of shading. Resolving
both to black painted every cell in the document solid black.

This document uses `w:fill="auto"` on all 2308 of its shading elements, so expect **no** resolved
cell fills at all.

In [ ]:
def iter_cells(blocks):
    for b in blocks:
        if isinstance(b, TableBlock):
            for row in b.rows:
                for cell in row.cells:
                    yield cell
                    yield from iter_cells(cell.blocks)

fills = Counter(c.shading_fill for c in iter_cells(doc.blocks))
print(f"table cells          : {sum(fills.values())}")
print(f"resolved cell fills  : {dict(fills)}")

black = fills.get("000000", 0)
print(f"\nPASS - no cell is filled black" if not black else f"\nFAIL - {black} cells filled black")

### Check 4 — no page may be blank

The layout engine forced every table row to at least 14 pt. A single line of 10 pt Arial measures
11.5 pt, so each row of a long table was silently padded by ~2.5 pt; over the 40 rows of Table 2
that is nearly a page, which pushed two trailing empty paragraphs past the bottom margin. They
landed on a page of their own, and because both were empty that page rendered blank. Word sizes a
row to its tallest cell and applies no minimum beyond `w:trHeight`.

Word records its own page count in `docProps/app.xml` `<Pages>`: **88**. With the floor removed the
document lays out to 88 pages, and every `PAGEREF` anchor lands on the page Word recorded for it
— see Check 5. (Counting `w:lastRenderedPageBreak` markers instead suggests 83, but those markers
are stale; `word_truth.py` and `NOTES.md` explain why they are a hint and not a score.)

In [ ]:
blank = [pg.page_number for pg in layout_pages if not (pg.texts or pg.images or pg.rects)]
print(f"pages                : {len(layout_pages)}")
print(f"blank pages          : {blank or 'none'}")
print("\nPASS - every page has content" if not blank else f"\nFAIL - nothing drawn on {blank}")

### Check 5 — pagination must match Word's own record of it

A `.docx` carries Word's own layout of itself, so pagination is checkable with no Word install.
Word leaves three traces and they are not equally trustworthy — `word_truth.py` reads all three:

| Word's record | on this sample | vintage |
|---|---|---|
| `docProps/app.xml` `<Pages>` | 88 pages | last save |
| cached `PAGEREF` result of every TOC / ToT / ToF entry | 118 page-number anchors | last field update |
| `<w:lastRenderedPageBreak/>` markers | 83 pages | stale — predates the field update |

**Score against the `PAGEREF` anchors.** Each TOC, Table of Tables and Table of Figures entry
carries the page number Word resolved its heading or caption to, which gives one assertable anchor
per entry spread over the whole document. The markers go stale because Word only refreshes them
when it actually repaginates a region.

The anchor score is deliberately brittle in a useful way: one extra page anywhere early shifts every
later anchor, so a regression drops it from 118/118 to near zero rather than shaving off a point.


In [ ]:
import word_truth as wt

recorded = wt.recorded_page_count(DOCX)
targets = wt.pageref_targets(DOCX)
page_of = wt.our_page_of_label(layout_pages)

misses = [(t, page_of.get(t.target_label)) for t in targets
          if page_of.get(t.target_label) != t.word_page]

print(f"pages                : {len(layout_pages)}  (Word's <Pages>: {recorded})")
print(f"PAGEREF anchors      : {len(targets) - len(misses)}/{len(targets)} on Word's page")
for t, our in misses[:12]:
    print(f"  {t.entry_label:<10}{t.style_id:<16}Word={t.word_page:<4}ours={our}  {t.text[:34]}")

ok = not misses and recorded == len(layout_pages)
print()
print("PASS - page count and every anchor match Word's own record" if ok
      else "FAIL - see above")

## 5. LLM visual-grounding mode

`--ruler` draws the 0.5" tab ruler and tab grid lines, `--bboxes` outlines every laid-out element,
and `--json` writes the XPath → bounding-box map. Useful for checking indentation and tab stops
against the XML.

The overlays are expensive to eyeball across a long document, so this is a good place for the
`pages=` argument.

In [ ]:
grounding_out = PROJECT / "inspect_out_ruler"

written = render_docx_to_png(
    docx_path=DOCX, output_dir=grounding_out, prefix="page", dpi=DPI,
    show_ruler=True, show_bboxes=True, export_layout_json=True,
    pages="1,4",
)
for path in written:
    display(HTML(f"<div style='margin-top:14px;font:600 13px system-ui'>"
                 f"{path.stem.replace('_', ' ')} &mdash; ruler + bboxes</div>"))
    display(IPyImage(filename=str(path), width=900))

## 6. Triage helper — which paragraph styles carry hyperlinks

This is the query that located bug 1: it groups every `<w:hyperlink>` in `word/document.xml` by the
paragraph style around it, separating internal anchors (`w:anchor`, i.e. TOC entries and
cross-references) from external relationship links. A TOC-family style showing up under `anchor`
should render black; anything else is a real link and should be blue.

In [ ]:
import zipfile
from lxml import etree

W = "{http://schemas.openxmlformats.org/wordprocessingml/2006/main}"
root = etree.fromstring(zipfile.ZipFile(DOCX).read("word/document.xml"))

tally = Counter()
for p in root.iter(W + "p"):
    links = list(p.iter(W + "hyperlink"))
    if not links:
        continue
    ps = p.find(f"{W}pPr/{W}pStyle")
    sid = ps.get(W + "val") if ps is not None else None
    kind = "anchor" if any(h.get(W + "anchor") for h in links) else "external"
    tally[(sid, kind)] += 1

print(f"{'paragraph style':<20} {'link kind':<10} {'count':>5}   treated as TOC entry?")
for (sid, kind), n in tally.most_common():
    print(f"{str(sid):<20} {kind:<10} {n:>5}   {parser.is_toc_entry_style(sid)}")

## 7. Render a different file

Point `DOCX` at any other `.docx` and re-run from section 2 — or do it in one call here.

In [ ]:
other = PROJECT / "Table of Contents template example.docx"
other_out = PROJECT / "inspect_out_other"

if other.exists():
    op = render_docx_to_png(docx_path=other, output_dir=other_out, prefix="page", dpi=DPI)
    print(f"{len(op)} pages -> {other_out}")
    display(IPyImage(filename=str(other_out / "page_1.png"), width=760))
else:
    print(f"not found: {other}")